In [1]:
!pip install mlflow

!pip install --upgrade mlflow cachetools dagshub transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.2.19
    Uninstalling SQLAlchemy-1.2.19:
      Successfully uninstalled SQLAlchemy-1.2.19
ERROR: pip's dependency resolv

In [2]:
import os
from pyexpat import model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import time
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW 
from transformers import XLMRobertaModel, XLMRobertaTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_fscore_support
from tqdm.notebook import tqdm
import mlflow
import mlflow.pytorch
import dagshub

2026-01-14 13:10:54.222811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768396254.425263      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768396254.481349      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768396254.945308      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768396254.945340      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768396254.945342      55 computation_placer.cc:177] computation placer alr

In [ ]:
# --- 1. CONFIGURAZIONE AMBIENTE & MLFLOW (DAGsHub) ---
# Inserisci i tuoi dati DAGsHub qui
DAGSHUB_USER = ""
DAGSHUB_REPO = "AISE_labeling"
DAGSHUB_TOKEN = ""

os.environ['MLFLOW_TRACKING_USERNAME'] = 'matdavino'
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("")


2026/01/14 13:11:43 INFO mlflow.tracking.fluent: Experiment with name 'Training_parallel_oversample_andrea' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1d8bf1d37a5a45699ddc1414ba54275b', creation_time=1768396303219, experiment_id='12', last_update_time=1768396303219, lifecycle_stage='active', name='Training_parallel_oversample_andrea', tags={}>

In [ ]:
MODEL_NAME = 'xlm-roberta-base' 
MAX_LEN_MSG = 512               
MAX_MSG_PER_CONV = 120          
BATCH_SIZE = 2                 
EPOCHS = 15
WARMUP_EPOCHS = 3               
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MSG_LABEL_COLS = ['OVERRELIANCE_DECISION', 'AUTONOMY_LOW_SELF', 'EMOTIONAL_DEPENDENCE', 'PARASOCIAL_REL', 
                  'ANTHROPOMORPHISM_STRONG', 'ADDICTIVE_USE', 'SOCIAL_WITHDRAWAL', 'SKILL_ATROPHY', 
                  'ANXIETY_ESCALATION', 'CRISIS_RISK']
CONV_LABEL_COLS = [c + '_C' for c in MSG_LABEL_COLS]

dtype_dict_1 = {col: 'Int64' for col in MSG_LABEL_COLS}
dtype_dict_2 = {col: 'Int64' for col in CONV_LABEL_COLS}
dtype_dict_2['global_risk_flag'] = 'Int64'



In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, path='best_model.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss:
            self.counter += 1
            print(f'\n>>> EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    
    def save_checkpoint(self, model):
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), self.path)
        else:
            torch.save(model.state_dict(), self.path)
        mlflow.log_artifact('best_model.pt', artifact_path="modelli_salvati")
        print(">>> Modello sincronizzato su DagsHub!")
        print(f'>>> Modello migliorato salvato localmente come {self.path}')


In [ ]:
class HierarchicalDataset(Dataset):
    def __init__(self, msg_df, conv_df, tokenizer):
        self.tokenizer = tokenizer
        self.samples = []
        combined = pd.merge(msg_df, conv_df, on='conversation_id', how='left')
        grouped = combined.sort_values(['conversation_id', 'message_id']).groupby('conversation_id')
        
        for _, group in tqdm(grouped, desc="Preparazione Dataset"):
            has_labels = not pd.isna(group['global_risk_flag'].iloc[0])
            c_multi = group[CONV_LABEL_COLS].iloc[0].values.astype(int) if has_labels else np.array([-100]*10)
            c_bin = int(group['global_risk_flag'].iloc[0]) if has_labels else -100
            
            texts = group['text'].astype(str).tolist()[:MAX_MSG_PER_CONV]
            m_labels = [row[MSG_LABEL_COLS].values.astype(int) if row['user_role'] == 'user' else [-100]*10 
                        for _, row in group.head(MAX_MSG_PER_CONV).iterrows()]
            
            self.samples.append({'texts': texts, 'm_labels': m_labels, 'c_multi': c_multi, 'c_bin': c_bin, 'len': len(texts)})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = self.tokenizer(s['texts'], padding='max_length', truncation=True, max_length=MAX_LEN_MSG, return_tensors='pt')
        pad = MAX_MSG_PER_CONV - s['len']
        ids = torch.cat([enc['input_ids'], torch.zeros((pad, MAX_LEN_MSG), dtype=torch.long)])
        mask = torch.cat([enc['attention_mask'], torch.zeros((pad, MAX_LEN_MSG), dtype=torch.long)])
        
        m_labels_np = np.array(s['m_labels'])
        m_lab = torch.cat([torch.from_numpy(m_labels_np), torch.full((pad, 10), -100)])
        
        return {'ids': ids, 'mask': mask, 'm_labels': m_lab, 
                'c_multi': torch.tensor(s['c_multi']), 'c_bin': torch.tensor(s['c_bin']), 'actual_len': s['len']}


In [ ]:
class HierarchicalModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained(name)
        for p in self.encoder.parameters(): p.requires_grad = False
                
        h = self.encoder.config.hidden_size
        self.m_heads = nn.ModuleList([nn.Linear(h, 3) for _ in range(10)]) 
        self.proj = nn.Linear(h + 30, 256)
        self.gru = nn.GRU(256, 256, num_layers=1, batch_first=True, bidirectional=True)
        self.c_multi_heads = nn.ModuleList([nn.Linear(512, 3) for _ in range(10)])
        self.c_bin_head = nn.Linear(512, 2) 

    def forward(self, ids, mask, lengths):
        B, S, L = ids.shape
        with torch.no_grad():
            emb = self.encoder(ids.view(-1, L), attention_mask=mask.view(-1, L)).last_hidden_state[:, 0, :]
        
        m_logits = torch.stack([head(emb) for head in self.m_heads], dim=1) 
        combined = self.proj(torch.cat([emb.view(B, S, -1), m_logits.view(B, S, -1)], dim=-1))
        gru_out, _ = self.gru(combined)
        final_state = gru_out[torch.arange(B), (lengths - 1).clamp(min=0)]
        
        return m_logits.view(B, S, 10, 3), torch.stack([head(final_state) for head in self.c_multi_heads], dim=1), self.c_bin_head(final_state)


In [ ]:
def get_class_weights(df, columns, num_classes=3):
    weights_dict = {}
    for col in columns:
        counts = df[col].value_counts().to_dict()
        total = sum(counts.values())
        
        weights = []
        for i in range(num_classes):
            count = counts.get(i, 0)
            w = total / (num_classes * count) if count > 0 else 1.0
            weights.append(w)
            
        weights_dict[col] = torch.tensor(weights, dtype=torch.float).to(DEVICE)
    return weights_dict


In [ ]:
def evaluate(model, loader, criteria_msgs, criteria_conv_multi, criterion_global, device, is_warmup=False):
    model.eval()
    val_loss, valid_steps = 0, 0
    
    pred_distribution = {0: 0, 1: 0, 2: 0}

    y_true_msgs = [[] for _ in range(10)]
    y_pred_msgs = [[] for _ in range(10)]
    y_true_convs = [[] for _ in range(10)]
    y_pred_convs = [[] for _ in range(10)]
    y_true_bin, y_pred_bin = [], []
    
    with torch.no_grad():
        for b in loader:
            m_log, c_m_log, c_b_log = model(b['ids'].to(device), b['mask'].to(device), b['actual_len'].to(device))
            
            # --- Calcolo Loss PESATA (stessa del training) ---
            loss_step = sum(criteria_msgs[i](m_log[:,:,i,:].reshape(-1,3), b['m_labels'][:,:,i].to(device).reshape(-1)) for i in range(10))
            
            if not is_warmup:
                if not (b['c_multi'] == -100).all():
                    loss_step += sum(criteria_conv_multi[i](c_m_log[:, i, :], b['c_multi'][:, i].to(device)) for i in range(10))
                if not (b['c_bin'] == -100).all():
                    loss_step += criterion_global(c_b_log, b['c_bin'].to(device))
            
            val_loss += loss_step.item()
            valid_steps += 1
            # --- RACCOLTA DATI MESSAGGI ---
            m_preds = torch.argmax(m_log, dim=-1).cpu().numpy()
            m_targets = b['m_labels'].numpy()

            for i in range(10):
                t, p = m_targets[:,:,i].flatten(), m_preds[:,:,i].flatten()
                mask = t != -100
                y_true_msgs[i].extend(t[mask])
                y_pred_msgs[i].extend(p[mask])
                
                # Conta le occorrenze delle classi predette
                if len(p[mask]) > 0:
                    unique, counts = np.unique(p[mask], return_counts=True)
                    for val, count in zip(unique, counts):
                        pred_distribution[int(val)] += count

                        
            # --- RACCOLTA DATI CONVERSAZIONE (MULTI-LABEL) ---
            # c_m_log shape: (Batch, 10, 3)
            c_m_preds = torch.argmax(c_m_log, dim=-1).cpu().numpy()
            c_m_targets = b['c_multi'].numpy()
            for i in range(10):
                t, p = c_m_targets[:, i], c_m_preds[:, i]
                mask = t != -100
                y_true_convs[i].extend(t[mask]); y_pred_convs[i].extend(p[mask])

            # --- RACCOLTA DATI GLOBALE (BINARY) ---
            p_bin = torch.argmax(c_b_log, dim=-1).cpu().numpy()
            t_bin = b['c_bin'].numpy()
            mask_bin = t_bin != -100
            y_pred_bin.extend(p_bin[mask_bin]); y_true_bin.extend(t_bin[mask_bin])

    # --- CALCOLO F1-SCORES ---
    f1_msg_heads = [f1_score(y_true_msgs[i], y_pred_msgs[i], average='macro', zero_division=0) for i in range(10)]
    f1_conv_heads = [f1_score(y_true_convs[i], y_pred_convs[i], average='macro', zero_division=0) for i in range(10)]
    f1_globale = f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0) if y_true_bin else 0    
    
    return val_loss / max(1, valid_steps), f1_globale, f1_msg_heads, f1_conv_heads, pred_distribution


In [ ]:
def run_pipeline():
    # Caricamento Dataset (Assicurati che i nomi dei file siano corretti)
    df_msg, df_conv = pd.read_csv('/kaggle/input/dataset_oversampled_pesante.csv', encoding='utf-8-sig' ,dtype=dtype_dict_1), pd.read_csv('/kaggle/input/dataset_appoggio_ridotto_GPT_etichettato_new_popolato.csv',encoding='utf-8-sig' , dtype=dtype_dict_2)
    tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
    
    # 1. Identifichiamo tutti gli ID unici e associamoli al loro stato in df_conv
    df_all_ids = pd.DataFrame({'conversation_id': df_msg['conversation_id'].unique()})

    # Uniamo con df_conv per recuperare il flag globale (dove esiste)
    df_strat = pd.merge(df_all_ids, df_conv[['conversation_id', 'global_risk_flag']], 
                        on='conversation_id', how='left')
    
    # 2. Creiamo la chiave di stratificazione
    df_strat['is_full_conv'] = df_strat['global_risk_flag'].notna().astype(int)
    df_strat['risk_val'] = df_strat['global_risk_flag'].fillna(-1).astype(int)
    df_strat['strat_key'] = df_strat['is_full_conv'].astype(str) + "_" + df_strat['risk_val'].astype(str)
    
    # --- CORREZIONE PER CLASSI RARE ---
    # Controlliamo quali classi hanno meno di 2 membri (causano l'errore di split)
    counts = df_strat['strat_key'].value_counts()
    rare_keys = counts[counts < 2].index
    
    # Se una classe è rarissima, la raggruppiamo in una categoria comune 'rare'
    df_strat['strat_key_robust'] = df_strat['strat_key'].apply(
        lambda x: 'rare_cases' if x in rare_keys else x
    )
    
    ids = df_strat['conversation_id'].values
    strat_labels = df_strat['strat_key_robust'].values
    
    # 3. Eseguiamo il Primo Split (70% Train, 30% Temp)
    tr_ids, temp_ids, _, temp_labels = train_test_split(
        ids, strat_labels, test_size=0.3, random_state=42, stratify=strat_labels
    )
    
    # --- CORREZIONE PER IL SECONDO SPLIT ---
    # Verifichiamo se nel set temporaneo (30%) sono rimaste classi con 1 solo membro
    temp_counts = pd.Series(temp_labels).value_counts()
    # Se ci sono classi con meno di 2 membri nel set temporaneo, disabilitiamo 
    # la stratificazione per l'ultimo split (15% Val / 15% Test) per evitare il crash
    strat_param = temp_labels if (temp_counts < 2).sum() == 0 else None
    
    if strat_param is None:
        print("(!) Alert: Stratificazione disabilitata per l'ultimo split a causa della scarsità di campioni in alcune classi.")
    
    # Secondo split: 15% Val, 15% Test
    vl_ids, ts_ids = train_test_split(
        temp_ids, test_size=0.5, random_state=42, stratify=strat_param
    )

    loaders = {
        'train': DataLoader(HierarchicalDataset(df_msg[df_msg['conversation_id'].isin(tr_ids)], df_conv, tokenizer), batch_size=BATCH_SIZE, shuffle=True),
        'val': DataLoader(HierarchicalDataset(df_msg[df_msg['conversation_id'].isin(vl_ids)], df_conv, tokenizer), batch_size=BATCH_SIZE),
        'test': DataLoader(HierarchicalDataset(df_msg[df_msg['conversation_id'].isin(ts_ids)], df_conv, tokenizer), batch_size=BATCH_SIZE)
    }

    # --- MODIFICA QUI: Inizializzazione Modello ---
    model = HierarchicalModel(MODEL_NAME) # Non spostare ancora su DEVICE
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model) # Parallelizza il modello
    
    model = model.to(DEVICE) 
    base_model = model.module if isinstance(model, nn.DataParallel) else model

    optimizer = AdamW([
        {'params': base_model.encoder.parameters(), 'lr': 1e-5}, # LR basso per l'encoder
        {'params': base_model.gru.parameters(), 'lr': 1e-4},
        {'params': base_model.proj.parameters(), 'lr': 1e-4},
        {'params': base_model.m_heads.parameters(), 'lr': 1e-4},
        {'params': base_model.c_multi_heads.parameters(), 'lr': 1e-4},
        {'params': base_model.c_bin_head.parameters(), 'lr': 1e-4},
    ])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)
    
    early_stopping = EarlyStopping(patience=3)

    # 1. Calcolo pesi per Messaggi (10 teste, 3 classi)
    msg_weights = get_class_weights(df_msg, MSG_LABEL_COLS, num_classes=3)
    for col in MSG_LABEL_COLS:
        msg_weights[col][1] *= 2.5  # Aumenta peso Classe 1
        msg_weights[col][2] *= 5.0  # Aumenta peso Classe 2
        
    # 2. Calcolo pesi per Conversazione Multi-label (10 teste, 3 classi)
    conv_multi_weights = get_class_weights(df_conv, CONV_LABEL_COLS, num_classes=3)

    for col in CONV_LABEL_COLS:
        conv_multi_weights[col][1] *= 2.5  # Aumenta peso Classe 1
        conv_multi_weights[col][2] *= 5.0  # Aumenta peso Classe 2
        
    # 3. Calcolo pesi per Global Risk (1 testa, 2 classi)
    global_weights = get_class_weights(df_conv, ['global_risk_flag'], num_classes=2)['global_risk_flag']

    # Definiamo i criteri specifici
    criterion_msgs = [nn.CrossEntropyLoss(weight=msg_weights[col], ignore_index=-100) for col in MSG_LABEL_COLS]
    criterion_conv_multi = [nn.CrossEntropyLoss(weight=conv_multi_weights[col], ignore_index=-100) for col in CONV_LABEL_COLS]
    criterion_global = nn.CrossEntropyLoss(weight=global_weights, ignore_index=-100)

    with mlflow.start_run(run_name="Matteo_Hierarchical_Multilingual_Model_2"):
        # 2. LOG DEI PESI come Artefatto (per ispezione)
        weights_to_log = {col: msg_weights[col].tolist() for col in MSG_LABEL_COLS}
        mlflow.log_dict(weights_to_log, "metadata/class_weights.json")
        
        # Log di un esempio di peso per capire l'ordine di grandezza (es. per la prima label)
        mlflow.log_param("weight_class_0", weights_to_log[MSG_LABEL_COLS[0]][0])
        mlflow.log_param("weight_class_2", weights_to_log[MSG_LABEL_COLS[0]][2])
        for epoch in range(EPOCHS):
            model.train()
            is_warmup = epoch < WARMUP_EPOCHS
            
            # --- FIX PER DATAPARALLEL ---
            # Se il modello è in parallelo, accediamo a .module, altrimenti usiamo il modello diretto
            if isinstance(model, nn.DataParallel):
                model_to_freeze = model.module
            else:
                model_to_freeze = model

            # Double Freeze Logic (usando model_to_freeze invece di model)
            for module in [model_to_freeze.proj, model_to_freeze.gru, model_to_freeze.c_multi_heads, model_to_freeze.c_bin_head]:
                for p in module.parameters(): p.requires_grad = not is_warmup

           # --- 2. INTEGRAZIONE SBLOCCO ENCODER ---
            if not is_warmup:
                # Sblocchiamo gli ultimi 3 strati dell'encoder (fine-tuning)
                # XLM-RoBERTa ha 12 layer (da 0 a 11)
                for p in model_to_freeze.encoder.encoder.layer[-3:].parameters():
                    p.requires_grad = True
                
                # È fondamentale sbloccare anche il pooler se lo usi, 
                # per permettere ai gradienti di fluire correttamente verso l'encoder
                if hasattr(model_to_freeze.encoder, 'pooler'):
                    for p in model_to_freeze.encoder.pooler.parameters():
                        p.requires_grad = True
                
                print(f">>> Epoca {epoch+1}: Encoder sbloccato (ultimi 3 layer)")
            else:
                # Durante il warmup assicuriamoci che l'encoder sia completamente frozen
                for p in model_to_freeze.encoder.parameters():
                    p.requires_grad = False
                print(f">>> Epoca {epoch+1}: Warmup in corso (Encoder frozen)")
            
            train_loss = 0
            train_steps = 0
            pbar = tqdm(loaders['train'], desc=f"Epoca {epoch+1}/{EPOCHS}")
            for b in pbar:
                optimizer.zero_grad()
                m_log, c_m_log, c_b_log = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE), b['actual_len'].to(DEVICE))
                
                # 1. Loss Messaggi
                l_m = sum(criterion_msgs[i](m_log[:,:,i,:].reshape(-1,3), b['m_labels'][:,:,i].to(DEVICE).reshape(-1)) for i in range(10))

                l_c = torch.tensor(0.0).to(DEVICE)
                if not is_warmup:
                    # 2. Loss Conversazione Multi-label
                    if not (b['c_multi'] == -100).all():
                        l_c += sum(criterion_conv_multi[i](c_m_log[:, i, :], b['c_multi'][:, i].to(DEVICE)) for i in range(10))
                    # 3. Loss Globale Binaria
                    if not (b['c_bin'] == -100).all():
                        l_c += criterion_global(c_b_log, b['c_bin'].to(DEVICE))

                loss = l_m + l_c                
                # 3. Controllo finale prima di aggiornare i pesi
                if not torch.isnan(loss) and loss > 0:
                    loss.backward()
                    optimizer.step()
                    train_loss += loss.item()
                    train_steps += 1 # <--- INCREMENTA SOLO SE IL BATCH È VALIDO
                else:
                    # Se la loss è nan, saltiamo il batch senza rovinare il modello
                    optimizer.zero_grad() 
                    
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})


            v_loss, v_f1, f1_msgs, f1_convs, p_dist = evaluate(model, loaders['val'], criterion_msgs, criterion_conv_multi, criterion_global, DEVICE, is_warmup=is_warmup)
            if not is_warmup:
                scheduler.step(v_loss)  # Lo scheduler analizza la loss e decide se intervenire
            avg_train_loss = train_loss / max(1, train_steps)

            total_preds = sum(p_dist.values())
            if total_preds > 0:
                mlflow.log_metric("pred_ratio_class_0", p_dist[0] / total_preds, step=epoch)
                mlflow.log_metric("pred_ratio_class_1", p_dist[1] / total_preds, step=epoch)
                mlflow.log_metric("pred_ratio_class_2", p_dist[2] / total_preds, step=epoch)
                
                # Log del numero assoluto per debugging rapido
                mlflow.log_metric("pred_count_class_2", p_dist[2], step=epoch)

            # Log su MLflow
            mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
            mlflow.log_metric("val_loss", v_loss, step=epoch)
            mlflow.log_metric("val_f1_global", v_f1, step=epoch)
            
            # 2. Log F1 Messaggi (M)
            for i, name in enumerate(MSG_LABEL_COLS):
                mlflow.log_metric(f"f1_M_{name.lower()}", f1_msgs[i], step=epoch)

            # 3. Log F1 Conversazione (C)
            for i, name in enumerate(MSG_LABEL_COLS):
                mlflow.log_metric(f"f1_C_{name.lower()}", f1_convs[i], step=epoch)

            print(f"\n[EPOCA {epoch+1}] T-Loss: {avg_train_loss:.4f} | V-Loss: {v_loss:.4f}\n")
            print(f"Epoca {epoch+1}: Predizioni Classe 2: {p_dist[2]} | F1 Macro: {v_f1:.4f}\n")
            
            if not is_warmup:
                early_stopping(v_loss, model)
                if early_stopping.early_stop: break

        # --- Fine del ciclo delle epoche ---
        print("\n>>> Caricamento del miglior modello per il TEST FINALE...")
        best_weights = torch.load('best_model.pt')
        # Se il modello è attualmente in DataParallel, carichiamo i pesi nel sotto-modulo
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(best_weights)
        else:
            model.load_state_dict(best_weights)

        # Valutazione sul Test Set (is_warmup=False perché vogliamo testare tutto il modello)
        ts_loss, ts_f1, ts_f1_msgs, ts_f1_convs, ts_p_dist = evaluate(
            model, loaders['test'], criterion_msgs, criterion_conv_multi, criterion_global, DEVICE, is_warmup=False
        )
                    
        for i, name in enumerate(MSG_LABEL_COLS):
            mlflow.log_metric(f"f1_M_T_{name.lower()}", ts_f1_msgs[i])

        for i, name in enumerate(MSG_LABEL_COLS):
            mlflow.log_metric(f"f1_C_T_{name.lower()}", ts_f1_convs[i])

        # 3. Log delle Metriche di Test su MLflow
        mlflow.log_metric("final_test_loss", ts_loss)
        mlflow.log_metric("final_test_f1_macro_global", ts_f1)

        # Log distribuzione predizioni sul test set per confronto
        total_ts_preds = sum(ts_p_dist.values())
        if total_ts_preds > 0:
            mlflow.log_metric("test_pred_ratio_class_2", ts_p_dist[2] / total_ts_preds)

        print(f"\n================ TEST FINALE ================")
        print(f"Test Loss: {ts_loss:.4f} | Test F1 Macro: {ts_f1:.4f}")
        print(f"Distribuzione Predizioni Test: {ts_p_dist}")
        print(f"=============================================")

In [16]:
run_pipeline()

(!) Alert: Stratificazione disabilitata per l'ultimo split a causa della scarsità di campioni in alcune classi.


Preparazione Dataset:   0%|          | 0/798 [00:00<?, ?it/s]

Preparazione Dataset:   0%|          | 0/171 [00:00<?, ?it/s]

Preparazione Dataset:   0%|          | 0/172 [00:00<?, ?it/s]

Using 2 GPUs!
>>> Epoca 1: Encoder sbloccato (ultimi 3 layer)


Epoca 1/10:   0%|          | 0/200 [00:00<?, ?it/s]

🏃 View run Matteo_Hierarchical_Multilingual_Model_2 at: https://dagshub.com/luigi.fienga18/AISE_labeling.mlflow/#/experiments/12/runs/1c7987c031f44105bf353f4437534e7a
🧪 View experiment at: https://dagshub.com/luigi.fienga18/AISE_labeling.mlflow/#/experiments/12


KeyboardInterrupt: 